## Проверка A/B-групп

В `views.csv` записаны выдачи рекомендаций с `user_id` и `exp_group`. Сначала проверяю, не попадал ли один пользователь в обе группы. Таких пользователей исключаю из анализа.

Затем считаю число и долю уникальных пользователей в каждой группе. Баланс распределения 50/50 проверяю двусторонним биномиальным тестом при уровне значимости 0,05.


In [1]:
import pandas as pd 
views = pd.read_csv("views.csv")
likes = pd.read_csv("likes.csv")

In [2]:
views.head()

,user_id,exp_group,recommendations,timestamp
0,128381,control,[3644 4529 4704 5294 4808],1654030803
1,146885,test,[1399 1076 797 7015 5942],1654030811
2,50948,test,[2315 3037 1861 6567 4093],1654030825
3,37703,test,[2842 1949 162 1588 6794],1654030826
4,14661,test,[2395 5881 5648 3417 673],1654030829


In [3]:
views.columns

Index(['user_id', 'exp_group', 'recommendations', 'timestamp'], dtype='str')

In [4]:
user_groups = views.groupby('user_id')['exp_group'].nunique()
user_groups.value_counts()

exp_group
1    65009
2        4
Name: count, dtype: int64

In [5]:
users_both_groups = user_groups[user_groups > 1]

bad_users = users_both_groups.index

mask = ~views["user_id"].isin(bad_users)

views = views[mask].copy()

In [6]:
views.groupby("user_id")["exp_group"].nunique().value_counts()

exp_group
1    65009
Name: count, dtype: int64

In [7]:
user_group = (
    views[["user_id", "exp_group"]]
    .drop_duplicates()
)
user_group["exp_group"].value_counts()

exp_group
test       32659
control    32350
Name: count, dtype: int64

In [8]:
user_group["exp_group"].value_counts(normalize=True)

exp_group
test       0.502377
control    0.497623
Name: proportion, dtype: float64

In [9]:
from scipy.stats import binomtest

control_count = user_group['exp_group'].value_counts()['control']
test_count = user_group['exp_group'].value_counts()['test']

total_count = len(user_group)

result = binomtest(control_count, n = total_count, p = 0.5, alternative = 'two-sided')
result.pvalue

np.float64(0.22705015636147555)

## Активность пользователей

Для каждого пользователя, получившего хотя бы одну выдачу рекомендаций, считаю число лайков из `likes.csv`. Если лайков нет, записываю 0. Затем рассчитываю долю пользователей, поставивших хотя бы один лайк.

На этом этапе лайки не сопоставляются с конкретными выдачами: мы оцениваем общую активность пользователей за период эксперимента.


In [10]:
likes.head()

,user_id,post_id,timestamp
0,128381,4704,1654030804
1,146885,1399,1654030816
2,50948,2315,1654030828
3,14661,673,1654030831
4,37703,1588,1654030833


In [11]:
experiment_users = views["user_id"].drop_duplicates()
user_likes = likes['user_id'].value_counts()

experiment_users = pd.DataFrame({
    'user_id' : views["user_id"].drop_duplicates()
})
experiment_users["likes"] = (
    experiment_users["user_id"]
    .map(user_likes)
    .fillna(0)
    .astype(int)
)
experiment_users.head()

,user_id,likes
0,128381,7
1,146885,4
2,50948,5
3,37703,6
4,14661,11


In [12]:
share_liked = (experiment_users["likes"] > 0).mean()
share_liked

np.float64(0.8947530341952653)

## Сравнение пользовательских метрик

Проверяю две метрики: долю пользователей с хотя бы одним лайком и число лайков на пользователя.

Для долей использую критерий χ² без поправки Йейтса (эквивалент двухстороннего z-критерия для двух долей). Для числа лайков применяю критерий Манна — Уитни, поскольку распределение счётной метрики может быть асимметричным. Уровень значимости — 0,05.

Критерий Манна — Уитни проверяет различие распределений, а не непосредственно разницу средних. Поэтому направление эффекта нужно оценивать по описательной статистике групп.


In [13]:
experiment_users = experiment_users.merge(
    user_group,
    on="user_id",
    how="left"
)
experiment_users

,user_id,likes,exp_group
0,128381,7,control
1,146885,4,test
2,50948,5,test
3,37703,6,test
4,14661,11,test
...,...,...,...
65004,80500,1,test
65005,149686,1,test
65006,3615,1,control
65007,119630,1,test


In [14]:
control_likes = experiment_users.loc[
    experiment_users["exp_group"] == "control",
    "likes"
]

test_likes = experiment_users.loc[
    experiment_users["exp_group"] == "test",
    "likes"
]

from scipy.stats import mannwhitneyu

result = mannwhitneyu(control_likes, test_likes, alternative='two-sided')
result

MannwhitneyuResult(statistic=np.float64(518358073.0), pvalue=np.float64(2.9585062792441953e-05))

In [15]:
experiment_users.groupby("exp_group").agg(
    users=("user_id", "size"),
    mean_likes=("likes", "mean"),
    median_likes=("likes", "median"),
    share_liked=("likes", lambda x: (x > 0).mean())
)

,users,mean_likes,median_likes,share_liked
exp_group,,,,
control,32350,3.487079,3.0,0.891314
test,32659,3.592578,3.0,0.898160


In [16]:
# Доля пользователей хотя бы с одним лайком — бинарная метрика.
# Сравниваем две независимые доли, а не повторяем тест для числа лайков.
from scipy.stats import chi2_contingency, norm

liked_control = (control_likes > 0).sum()
liked_test = (test_likes > 0).sum()

share_liked_control = liked_control / len(control_likes)
share_liked_test = liked_test / len(test_likes)

contingency = [
    [liked_control, len(control_likes) - liked_control],
    [liked_test, len(test_likes) - liked_test],
]

# Хи-квадрат без поправки Йейтса эквивалентен квадрату z-критерия для двух долей.
chi2, pvalue_share, _, _ = chi2_contingency(contingency, correction=False)

share_diff = share_liked_test - share_liked_control
standard_error = (
    share_liked_control * (1 - share_liked_control) / len(control_likes)
    + share_liked_test * (1 - share_liked_test) / len(test_likes)
) ** 0.5
ci_low = share_diff - norm.ppf(0.975) * standard_error
ci_high = share_diff + norm.ppf(0.975) * standard_error

print(f"Control: {share_liked_control:.2%}")
print(f"Test: {share_liked_test:.2%}")
print(f"Разница (test - control): {share_diff * 100:+.2f} п.п.")
print(f"95% ДИ разницы: [{ci_low * 100:+.2f}; {ci_high * 100:+.2f}] п.п.")
print(f"p-value для долей: {pvalue_share:.6g}")

Control: 89.13%
Test: 89.82%
Разница (test - control): +0.68 п.п.
95% ДИ разницы: [+0.21; +1.16] п.п.
p-value для долей: 0.00445476


## Сопоставление выдач и лайков

В логах нет общего идентификатора показа и лайка, поэтому восстанавливаю соответствие по `user_id`, времени и `post_id`. Лайк считаю подходящим, если он произошёл не раньше выдачи, не позднее чем через час и относится к одному из рекомендованных постов.

Если один лайк подходит нескольким выдачам, отношу его к последней подходящей. Каждый лайк учитывается не более одного раза. Выдачи без сопоставленных лайков сохраняю — они тоже входят в знаменатель метрики.

Результат — **доля выдач, после которых найден хотя бы один лайк рекомендованного поста**. Это приближённая метрика по лайкам, а не точный CTR кликов: непосредственной связи лайка с конкретной выдачей нет.


In [17]:
# Каждому показу и каждому лайку задаём собственный ID.
views = views.reset_index(drop=True)
views["view_id"] = views.index

likes = likes.reset_index(drop=True).copy()
likes["like_id"] = likes.index

print("Показов:", len(views))
print("Лайков:", len(likes))

Показов: 193268
Лайков: 230176


In [18]:
views_likes = views.merge(
    likes,
    on="user_id",
    how="left",
    suffixes=("_view", "_like")
)
print("После join:", len(views_likes))

После join: 1016889


In [19]:
# Лайк должен быть поставлен после показа
views_likes.loc[
    views_likes["timestamp_like"] < views_likes["timestamp_view"],
    "post_id"
] = None

print("После проверки времени:", len(views_likes))

После проверки времени: 1016889


In [20]:
# Разница между показом и лайком
views_likes["time_diff"] = (
    views_likes["timestamp_like"]
    - views_likes["timestamp_view"]
)
# Если лайк позже чем через час — считаем, что лайка нет
views_likes.loc[
    views_likes["time_diff"] > 3600,
    "post_id"
] = None

print("После ограничения 1 час:", len(views_likes))
print(
    "Слишком поздних лайков:",
    (views_likes["time_diff"] > 3600).sum()
)

После ограничения 1 час: 1016889
Слишком поздних лайков: 385059


In [21]:
# Преобразуем рекомендации в список
views_likes["recommendations"] = views_likes["recommendations"].apply(
    lambda x: list(map(int, x.strip("[]").split()))
    if isinstance(x, str)
    else []
)

In [22]:
# Лайк должен относиться к посту,
# который был в рекомендациях этого показа
views_likes.loc[
    ~views_likes.apply(
        lambda row: (
            pd.notna(row["post_id"])
            and row["post_id"] in row["recommendations"]
        ),
        axis=1
    ),
    "post_id"
] = None

print("После проверки рекомендаций:", len(views_likes))

После проверки рекомендаций: 1016889


In [23]:
# Один и тот же лайк может подходить нескольким показам пользователя.
# Относим лайк только к самому позднему подходящему показу.
valid_likes = views_likes.dropna(subset=["like_id", "post_id"]).copy()
valid_likes = (
    valid_likes
    .sort_values(["like_id", "timestamp_view", "view_id"])
    .drop_duplicates(subset="like_id", keep="last")
)

# Для HitRate достаточно уникальных пар «показ — лайкнутый пост».
unique_pairs = valid_likes[["view_id", "post_id"]].drop_duplicates()

# Возвращаем показы без лайков, чтобы они оставались в знаменателе.
views_likes = views[["view_id", "user_id", "exp_group"]].merge(
    unique_pairs,
    on="view_id",
    how="left",
    validate="one_to_many",
)

print("Уникальных сопоставленных лайков:", valid_likes["like_id"].nunique())
print("Строк после уникализации:", len(views_likes))
print("Уникальных показов:", views_likes["view_id"].nunique())

Уникальных сопоставленных лайков: 230137
Строк после уникализации: 285555
Уникальных показов: 193268


In [24]:
# Для каждого показа — ровно один показатель hit: 0 или 1.
view_has_like = (
    views_likes
    .groupby("view_id")["post_id"]
    .apply(lambda posts: posts.notna().any())
)

hitrate = view_has_like.mean()
print(f"Доля показов с хотя бы одним лайком: {hitrate:.2%}")

Доля показов с хотя бы одним лайком: 71.33%


In [25]:
print("Исходное число показов:", len(views))
print("Число уникальных показов после обработки:", views_likes["view_id"].nunique())

# Каждый показ должен остаться в результате, включая показы без лайков.
assert views_likes["view_id"].nunique() == len(views)
assert view_has_like.index.nunique() == len(views)
assert valid_likes["like_id"].is_unique  # каждый лайк приписан максимум одному показу

Исходное число показов: 193268
Число уникальных показов после обработки: 193268


## Сравнение метрики по 100 бакетам

Сначала получаю одну строку на выдачу: `hit = 1`, если нашёлся хотя бы один подходящий лайк, иначе `0`. Так выдачи с несколькими лайками не получают дополнительного веса.

Один пользователь может получить несколько выдач, поэтому нельзя считать их независимыми наблюдениями. Разделяю пользователей на 100 бакетов по `user_id % 100`: все выдачи одного пользователя оказываются в одном бакете. Для каждой группы вычисляю долю успешных выдач внутри каждого бакета и сравниваю бакетные значения двусторонним t-критерием Уэлча при уровне значимости 0,05.

Общую долю успешных выдач в группе считаю непосредственно по показам, а не как невзвешенное среднее бакетов.


In [26]:
# Одна строка = один показ, даже если у него несколько лайков.
view_level = views[["view_id", "user_id", "exp_group"]].merge(
    view_has_like.rename("hit"),
    left_on="view_id",
    right_index=True,
    how="left",
    validate="one_to_one",
)
assert view_level["hit"].notna().all()
view_level["hit"] = view_level["hit"].astype(int)
view_level["bucket"] = view_level["user_id"] % 100

bucket_hitrate = (
    view_level
    .groupby(["exp_group", "bucket"], as_index=False)
    .agg(hit=("hit", "mean"), impressions=("hit", "size"))
)

control = bucket_hitrate.loc[
    bucket_hitrate["exp_group"] == "control", "hit"
]
test = bucket_hitrate.loc[
    bucket_hitrate["exp_group"] == "test", "hit"
]

# Общий HitRate считаем по показам, а не как среднее бакетных средних.
group_hitrate = view_level.groupby("exp_group")["hit"].mean()

In [27]:
from scipy.stats import ttest_ind

# Критерий Уэлча допускает разную дисперсию бакетных метрик.
stat, pvalue = ttest_ind(control, test, equal_var=False)

print(f"Control HitRate по показам: {group_hitrate['control']:.2%}")
print(f"Test HitRate по показам: {group_hitrate['test']:.2%}")
print(f"Разница (test - control): {(group_hitrate['test'] - group_hitrate['control']) * 100:+.2f} п.п.")
print(f"Бакетов: control={len(control)}, test={len(test)}")
print(f"p-value бакетного теста: {pvalue:.6g}")

if pvalue < 0.05:
    print("Различие статистически значимо на уровне 0.05 (бакетный тест)")
else:
    print("Статистически значимое различие на уровне 0.05 не обнаружено")

Control HitRate по показам: 70.67%
Test HitRate по показам: 71.98%
Разница (test - control): +1.32 п.п.
Бакетов: control=100, test=100
p-value бакетного теста: 8.08559e-08
Различие статистически значимо на уровне 0.05 (бакетный тест)


## Результаты и выводы

- **Распределение групп.** Удалены 4 пользователя, встречавшихся в обеих группах. Осталось 32 350 пользователей в Control и 32 659 в Test. Биномиальный тест дал `p = 0,227`: значимого отклонения от распределения 50/50 не обнаружено.
- **Пользователи с лайками.** В целом хотя бы один лайк поставили 89,48% пользователей. В Control — 89,13%, в Test — 89,82%. Разница около +0,68 процентного пункта, `p = 0,00445`.
- **Количество лайков.** Критерий Манна — Уитни показал различие между группами (`p ≈ 0,0000296`). Чтобы описать его направление и величину, нужно также сравнить среднее и медиану числа лайков в группах.
- **Доля выдач с лайком.** В целом хотя бы один лайк удалось сопоставить с 71,33% выдач. Control — 70,67%, Test — 71,98%; разница около +1,32 процентного пункта. Бакетный t-тест на 100 бакетах дал `p ≈ 8,09 × 10⁻⁸`. Различие статистически значимо при уровне 0,05.

**Ограничения.** Лайки сопоставляются с выдачами приближённо, без общего `impression_id`. Поэтому метрика показывает долю выдач с найденным лайком, а не точный CTR кликов. Анализ выполнен на предоставленных курсом логах учебного эксперимента; полученные цифры нельзя автоматически приписывать текущим моделям HGB и MLP. При одновременной проверке нескольких гипотез также стоит учитывать риск множественного тестирования.
